### 数据清洗

##### 数据导入  

In [41]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, f1_score
from sklearn.linear_model import LogisticRegression


In [2]:
data=pd.read_csv(r"d:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\我的数据-toy.csv")

In [ ]:
data

2. 数据概述


In [ ]:
data.describe().T

In [ ]:
data.info()

3.缺失值统计

In [ ]:
data.isnull().sum()

In [ ]:
df=data
null_number = sum(df.isnull().sum(axis=0))
null_number

In [ ]:
df[df.isnull().T.any()].head(null_number)

4. 缺失值处理
   

In [ ]:
null_index = df.isnull().T.any()
# 删除这些行
new_df = df[~null_index]
new_df

In [ ]:
df.fillna(0)

In [ ]:
# 选择数值列
numeric_columns = df.select_dtypes(include=[np.number]).columns
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].mean())
df

5. 异常值

In [ ]:
bank=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bank.csv")
bank

In [ ]:
px.box(bank,y="交易金额")

6. 基于 3 sigma 原则，删除 [avg - 3 * std, avg + 3 * std] 外的数据


In [ ]:

bank = bank[ bank["交易金额"]>0]
# 然后对这些数据进行正态分布拟合，求平均值和标准差
avg_val =bank["交易金额"].mean()
std_val =bank["交易金额"].std()
print(f"交易金额的平均值为：{avg_val}，标准差为：{std_val}")


In [ ]:
upper_bound = avg_val + 3 * std_val
lower_bound = avg_val - 3 * std_val
drop_index = (bank ["交易金额"] < lower_bound) | (bank["交易金额"] > upper_bound)
new_df_2 = bank[~drop_index]
# 来看看去异后箱线图
px.box(new_df_2,y="交易金额")


#### 离散化编码：数据分箱

In [ ]:
bank["交易金额_分箱后"] = pd.cut(bank["交易金额"], 3,labels=['低', '中', '高'])
bank

In [ ]:
bank["交易金额_分箱后_q"] = pd.qcut(bank["交易金额"], 3,labels=['低', '中', '高'])
bank

In [ ]:
from sklearn.cluster import KMeans 
clusterer = KMeans(3)  # 分成 3 堆
clusterer.fit(bank[["交易金额"]])
### 调包即可

In [ ]:
bank["交易金额_分箱后_聚类"] = clusterer.predict(bank[["交易金额"]])
print(bank.head(5))


1. 选择K个初始聚类中心
2. 重复:
   a. 将每个点分配到最近的聚类中心
   b. 重新计算每个簇的中心点
   直到聚类中心基本不变

In [ ]:
df

In [ ]:
def toDict(col):
    """
    这个函数的作用：生成替换字典，加入 col 列有两种值 A B
    则生成：{A: 0, B: 1}  即把 A 替换为 0 ， B 替换为 1
    """
    res = {}
    cnt = 0
    for e in col.unique():
        res[e] = cnt
        cnt += 1
    return res
# 生成列的替换字典
dict1 = toDict(df["性别"])
dict2 = toDict(df["职业"])
# 替换后结果
new_df = df.replace({"性别": dict1, "职业": dict2})
print(new_df.head(5))


In [22]:
new_df_3=pd.get_dummies(df, columns=["性别", "职业"],)

In [ ]:
new_df_3

#### 特征变换


In [24]:
mean, min, max, std = bank['交易金额'].agg(['mean', 'min', 'max', 'std'])

In [25]:
bank["交易金额_最大最小值归一化"] =bank['交易金额'].apply(lambda x:(x-mean)/(max-min))
bank['交易金额_标准化']=bank['交易金额'].apply(lambda x:(x-mean)/std)

In [ ]:
bank.head()

In [27]:
overdue=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\overdue.csv")
userinfo=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\userinfo.csv")


In [ ]:
overdue.head(3)

In [ ]:
overdue.isna().sum()

In [30]:
overdue.drop("Unnamed: 0",axis=1,inplace=True)

In [ ]:
userinfo.isna().sum()

In [32]:
userinfo.dropna(inplace=True)

In [ ]:
userinfo


In [34]:
dict_1 = toDict(userinfo['性别'])
dict_2 = toDict(userinfo['职业'])
dict_3 = toDict(userinfo['教育程度'])
dict_4 = toDict(userinfo['婚姻状态'])
dict_5 = toDict(userinfo['户口类型'])

##### 编码

In [ ]:
dict_1 = toDict(userinfo['性别'])
dict_2 = toDict(userinfo['职业'])
dict_3 = toDict(userinfo['教育程度'])
dict_4 = toDict(userinfo['婚姻状态'])
dict_5 = toDict(userinfo['户口类型'])

# 添加职业的one-hot编码
userinfo_with_dummies = pd.get_dummies(userinfo, columns=['职业'], prefix='职业')
# 生成哑变量，数据集，对哪一列编码，编码前面名词后缀

columns_to_encode = {'性别': dict_1, 
                    '职业':dict_2,
                    '教育程度': dict_3,
                    '婚姻状态': dict_4, 
                    '户口类型': dict_5}
userinfo_encoded = userinfo_with_dummies.replace(columns_to_encode)

In [36]:
df = pd.merge(userinfo_encoded, overdue, on='new_user_id', how='left')

In [ ]:
df

In [ ]:
# 定义特征列
feat_0 = ['性别', '职业_事业单位', '教育程度', '婚姻状态', '户口类型','职业_企业','职业_公职','职业_学生','职业_无业']

# 分割训练集和测试集
X_train, X_test, Y_train, Y_test = train_test_split(df[feat_0], df[['标签']], test_size=0.2)

# 逻辑回归模型
model = LogisticRegression()
columns_to_encode = {'性别': dict_1, 
                    '职业_事业单位':dict_2,
                    '教育程度': dict_3,
                    '婚姻状态': dict_4, 
                    '户口类型': dict_5}
model = LinearRegression()
model.fit(X_train, Y_train)
Y_pred_val = model.predict(X_test)
mse = mean_squared_error(Y_test, Y_pred_val)     
print("均方误差:", mse)